# Notebook 2: Transportation Dataset Exploratory Data Analysis & Cleaning
**Author:** Member 2
**Domain:** Transportation
**Dataset:** Automated Traffic Volume Counts (NYC Open Data `6fi9-q3ta` / `btm5-ppia`)

---

## 1. Dataset Overview & Selection Rationale
- **Dataset Title:** NYC Traffic Volume Counts (Historical / Automated)
- **Source:** NYC Department of Transportation (DOT) / NYC Open Data
- **API / Download URL:** `https://data.cityofnewyork.us/api/views/6fi9-q3ta/rows.csv?accessType=download`
- **Rationale:** Captures traffic counts across NYC roadway segments over time. Aggregating traffic volume by geographic boundary (ZIP code / Borough) and temporal units (Year / Month) enables cross-analysis with affordable housing developments.


In [6]:
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
from pathlib import Path

# sns.set_theme(style="whitegrid")

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TRAFFIC_FILE = RAW_DIR / "traffic_raw.csv"

# Load Traffic Dataset
if TRAFFIC_FILE.exists():
    df_traffic = pd.read_csv(TRAFFIC_FILE, low_memory=False)
else:
    # Sample or direct download fallback
    url = "https://data.cityofnewyork.us/api/views/6fi9-q3ta/rows.csv?accessType=download"
    df_traffic = pd.read_csv(url, low_memory=False, nrows=100000)

print("Raw Traffic Dataset Shape:", df_traffic.shape)
df_traffic.head()


Raw Traffic Dataset Shape: (42756, 31)


,id,segmentid,roadway_name,from,to,direction,date,_12_00_1_00_am,_1_00_2_00am,_2_00_3_00am,...,_2_00_3_00pm,_3_00_4_00pm,_4_00_5_00pm,_5_00_6_00pm,_6_00_7_00pm,_7_00_8_00pm,_8_00_9_00pm,_9_00_10_00pm,_10_00_11_00pm,_11_00_12_00am
0,1,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,2012-01-09T00:00:00.000,20.0,10.0,11.0,...,104.0,105.0,147.0,120.0,91.0,83.0,74.0,49.0,42.0,42.0
1,2,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,2012-01-10T00:00:00.000,21.0,16.0,8.0,...,102.0,98.0,133.0,131.0,95.0,73.0,70.0,63.0,42.0,35.0
2,3,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,2012-01-11T00:00:00.000,27.0,14.0,6.0,...,115.0,115.0,130.0,143.0,106.0,89.0,68.0,64.0,56.0,43.0
3,4,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,2012-01-12T00:00:00.000,22.0,7.0,7.0,...,71.0,127.0,122.0,144.0,122.0,76.0,64.0,58.0,64.0,43.0
4,5,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,2012-01-13T00:00:00.000,31.0,17.0,7.0,...,113.0,126.0,133.0,135.0,102.0,106.0,58.0,58.0,55.0,54.0


## 2. Exploratory Data Analysis (EDA) & Summary
Examine schema, missingness, time spans, and volume distributions.


In [7]:
print("--- Traffic Columns ---")
print(df_traffic.columns.tolist())

print("--- Missing Values ---")
print(df_traffic.isnull().sum())


--- Traffic Columns ---
['id', 'segmentid', 'roadway_name', 'from', 'to', 'direction', 'date', '_12_00_1_00_am', '_1_00_2_00am', '_2_00_3_00am', '_3_00_4_00am', '_4_00_5_00am', '_5_00_6_00am', '_6_00_7_00am', '_7_00_8_00am', '_8_00_9_00am', '_9_00_10_00am', '_10_00_11_00am', '_11_00_12_00pm', '_12_00_1_00pm', '_1_00_2_00pm', '_2_00_3_00pm', '_3_00_4_00pm', '_4_00_5_00pm', '_5_00_6_00pm', '_6_00_7_00pm', '_7_00_8_00pm', '_8_00_9_00pm', '_9_00_10_00pm', '_10_00_11_00pm', '_11_00_12_00am']
--- Missing Values ---
id                  0
segmentid           0
roadway_name        0
from                0
to                  0
direction           0
date                0
_12_00_1_00_am      4
_1_00_2_00am        4
_2_00_3_00am        4
_3_00_4_00am        4
_4_00_5_00am        4
_5_00_6_00am        4
_6_00_7_00am        4
_7_00_8_00am        4
_8_00_9_00am        4
_9_00_10_00am       4
_10_00_11_00am      3
_11_00_12_00pm      1
_12_00_1_00pm     253
_1_00_2_00pm      253
_2_00_3_00pm      253
_

## 3. Data Cleaning & Spatial-Temporal Feature Extraction
- Standardize column names (snake_case).
- Extract timestamps, `year`, and hourly/daily aggregate traffic metrics.
- Map segment location/Borough and clean missing traffic count values.


In [8]:
df_t_clean = df_traffic.copy()

# Column renaming normalization
df_t_clean.columns = [c.lower().replace(' ', '_') for c in df_t_clean.columns]

# Parse date/time
date_col = [c for c in df_t_clean.columns if 'date' in c or 'time' in c or 'beginning' in c]
if date_col:
    df_t_clean['datetime'] = pd.to_datetime(df_t_clean[date_col[0]], errors='coerce')
    df_t_clean['year'] = df_t_clean['datetime'].dt.year

# Volumetric count feature normalization
vol_col = [c for c in df_t_clean.columns if 'vol' in c or 'count' in c or 'pedestrian' in c]
if vol_col:
    df_t_clean['traffic_volume'] = pd.to_numeric(df_t_clean[vol_col[0]], errors='coerce').fillna(0)
else:
    df_t_clean['traffic_volume'] = 1

# Drop invalid records lacking date
df_t_clean = df_t_clean.dropna(subset=['year'])
df_t_clean['year'] = df_t_clean['year'].astype(int)

# Aggregate traffic at the Year level (and Borough/ZIP if available)
print("Cleaned Traffic Dataset Shape:", df_t_clean.shape)

df_t_clean.to_csv(PROCESSED_DIR / "traffic_cleaned.csv", index=False)
print("Saved cleaned traffic dataset to data/processed/traffic_cleaned.csv")


Cleaned Traffic Dataset Shape: (42756, 34)
Saved cleaned traffic dataset to data/processed/traffic_cleaned.csv


## 4. Initial EDA Visualizations
Trend of overall recorded traffic volumes across sample years.


In [9]:
# plt.figure(figsize=(10, 4))
# df_t_clean.groupby('year')['traffic_volume'].mean().plot(kind='line', marker='o', color='coral')

# (Plotting disabled because matplotlib is not installed)# plt.title('Average Traffic Volume by Year')
# plt.xlabel('Year')
# plt.ylabel('Mean Volume')
# plt.tight_layout()
# plt.show()
